# Life As An AI Engineer — Setup & Core Concepts

## 🎯 By the end of this session you will have

- Understood the course structure, the technical challenges, and what certification requires
- Met the cohort staff, your peer supporters, and your journey group
- **Set up and validated** your AI engineering development environment
- Applied AI-assisted coding practice using Claude Code

The third one is the deliverable. The rest you get by showing up; a working machine you
have to actually build, and this session exists so that you do it **now** rather than in
week 4 with an assignment due.

---

## 🏗️ Build | 🚢 Ship | 📤 Share

**This notebook is how you earn that deliverable.** The outcomes above are the session;
below is what the notebook itself covers, what counts as done, and what to post when it is.

### 🏗️ Build
- This repository understood as a **git** — upstream, origin, local — with the commands run live
- Your first LLM API call, and the roles that every LLM message is built from
- Five prompt-engineering patterns, each shown weak→strong so you can see *why* it matters
- The frontend/backend split that every app in this course will use

### 🚢 Ship
Get every cell in this notebook running on your machine — that is the validated environment,
proven rather than assumed: not a clever app, but a machine you can build on for the next ten weeks.

### 📤 Share
Post the output of Part 0 (your environment check) in the cohort channel. If something
failed, post that instead — a broken setup found in week 1 is cheap, and in week 4 is not.

---

> **This notebook assumes nothing is installed yet.** Part 0 runs without an API key,
> so you can check your environment before you spend a cent. Everything from Part 2
> onward needs a key.

## How to use this notebook

| | |
|---|---|
| **Where to run it** | From the `01_Life_As_An_AI_Engineer/` folder, with `.venv` selected as your kernel |
| **Setup** | `uv sync`, then `cp .env.example .env` and add your OpenAI key |
| **Order** | Top to bottom. Part 0 needs no key; Parts 2+ do |
| **If a cell fails** | Read the error, then check the "If it's blocked" section of this folder's `README.md` |

Full install instructions, if you have not done the one-click setup yet:
[Initial Setup](../00_Docs/Prerequisites/Initial_Setup/README.md).


## Contents

**[Part 0 — Does your environment actually work?](#part-0)**  
checks: 0.1 Python · 0.2 packages · 0.3 CLI tools · 0.4 API key · 0.5 installer layout · 0.6 installer's own check

**[Part 1 — Version control](#part-1)**  
[Why version control](#why-version-control) · [How — this repository is a git](#how-git)

**[Part 2 — The LLM API call](#part-2)**  
[Why call an API at all?](#why-api) · [The three roles](#three-roles) · [Setup — the client and two helpers](#api-setup)

**[Part 3 — Prompt engineering, five patterns](#part-3)**  
[1 Persona](#pattern-1) · [2 Few-shot](#pattern-2) · [3 Chain of thought](#pattern-3) · [4 Structured output](#pattern-4) · [5 Grounded context](#pattern-5)

**[Part 4 — Frontend and backend](#part-4)**  
[Why split them at all?](#why-split) · [How — no demo, by design](#how-no-demo)

**[🏗️ Activity — stack three patterns](#activity)** · **[✅ Done when](#done-when)**

---

<a id="part-0"></a>
# Part 0 — Does your environment actually work?

> ### Why this comes first
>
> Every hour lost in this course to "it works on my machine" is lost here, in week 1,
> to something small: a Python that is too old, a kernel pointing at the wrong
> interpreter, a key in the wrong file. The five cells below check each of those
> **separately**, so a failure tells you *which* thing is broken.
>
> This part needs no API key and costs nothing.

### 0.1 — Which Python is this notebook actually running?

This is the single most common setup failure in the course, and it is not obvious:
you can have a perfect `.venv` on disk and still have the notebook attached to your
*system* Python. The two paths below have to point into this folder's `.venv`.

If they don't: in VS Code, click the kernel picker (top right) and choose the
interpreter at `01_Life_As_An_AI_Engineer/.venv/bin/python`.


In [1]:
import sys, platform
from pathlib import Path

print(f"Python version   : {platform.python_version()}")
print(f"Executable       : {sys.executable}")
print(f"Prefix           : {sys.prefix}")
print()

# The course pins >=3.12,<3.15 (see pyproject.toml).
major, minor = sys.version_info[:2]
ok_version = (major == 3) and (12 <= minor < 15)
print(f"{'✅' if ok_version else '❌'} Version in range >=3.12,<3.15")

# Is this interpreter the one inside THIS folder's .venv?
in_venv = ".venv" in sys.prefix
here_venv = (Path.cwd() / ".venv").resolve()
is_local = Path(sys.prefix).resolve() == here_venv
print(f"{'✅' if in_venv else '❌'} Running inside a virtual environment")
print(f"{'✅' if is_local else '⚠️ '} That venv is this folder's .venv")
if in_venv and not is_local:
    print(f"     expected: {here_venv}")
    print(f"     actual  : {Path(sys.prefix).resolve()}")
    print("     -> switch your kernel, or you'll install packages into the wrong place")


Python version   : 3.14.2
Executable       : c:\Projects\AIE11\AIE11\01_Life_As_An_AI_Engineer\.venv\Scripts\python.exe
Prefix           : c:\Projects\AIE11\AIE11\01_Life_As_An_AI_Engineer\.venv

✅ Version in range >=3.12,<3.15
✅ Running inside a virtual environment
✅ That venv is this folder's .venv


### 0.2 — Are the packages importable?

`uv sync` installs them; this proves the *running kernel* can see them. A failure here
with a passing 0.1 almost always means you ran `uv sync` in a different folder.


In [2]:
import importlib

required = {
    "openai":   "the LLM client",
    "dotenv":   "reads your .env file (package name: python-dotenv)",
    "pydantic": "structured output schemas (Part 3)",
}

missing = []
for mod, why in required.items():
    try:
        m = importlib.import_module(mod)
        version = getattr(m, "__version__", "?")
        print(f"✅ {mod:<10} {version:<12} {why}")
    except ImportError:
        missing.append(mod)
        print(f"❌ {mod:<10} {'MISSING':<12} {why}")

if missing:
    print()
    print("Fix: run `uv sync` in 01_Life_As_An_AI_Engineer/, then restart the kernel.")


✅ openai     3.49.0       the LLM client
✅ dotenv     ?            reads your .env file (package name: python-dotenv)
✅ pydantic   2.14.3       structured output schemas (Part 3)


### 0.3 — Are the command-line tools there?

`git`, `uv`, `node` and `claude` are the tools you will use every week. We ask the
**shell**, not Python, because the shell is where you will actually type them.

A "not found" here almost never means the tool is missing. It means the tools are
installed but **not loaded into this terminal**. The course installer deliberately does
not touch every shell on your machine; you load it per terminal:

```bash
source ~/ai-eng/env.sh          # Mac/Linux
. $HOME\ai-eng\env.ps1          # Windows
```

That line goes in **every new terminal** you open for coursework. The one exception is
VS Code's `code` command, which is configured once and works everywhere.

> If you launched this notebook from a terminal that had not been loaded, the kernel
> inherited that environment — so load it, then **restart the kernel**.


In [3]:
import shutil, subprocess

def tool_version(name, args):
    """Return a one-line version string, or None if the tool isn't on PATH."""
    if shutil.which(name) is None:
        return None
    try:
        out = subprocess.run([name, *args], capture_output=True, text=True, timeout=15)
        return (out.stdout or out.stderr).strip().splitlines()[0]
    except Exception as exc:
        return f"(found, but failed to run: {exc})"

for name, args, needed in [
    ("git",    ["--version"], "required"),
    ("uv",     ["--version"], "required"),
    ("claude", ["--version"], "recommended"),
    ("node",   ["--version"], "needed later, for frontends"),
]:
    v = tool_version(name, args)
    mark = "✅" if v else ("❌" if needed == "required" else "⚠️ ")
    print(f"{mark} {name:<8} {v or 'not on PATH'}   ({needed})")


✅ git      git version 2.51.2.windows.1   (required)
✅ uv       uv 0.9.17 (2b5d65e61 2025-12-09)   (required)
✅ claude   2.1.29 (Claude Code)   (recommended)
✅ node     v25.9.0   (needed later, for frontends)


### 0.4 — Is your API key loaded?

Two separate things have to be true: the `.env` file exists, **and** this process read it.
We check both, and we never print the key itself — only its shape.

> ⚠️ **Never print or paste a full API key**, into a notebook output, a screenshot, or a
> help channel. Notebook outputs get committed to git. The check below shows only the
> prefix and length, which is enough to debug and useless to a thief.


In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

# This folder's .env takes precedence; the repo root is a fallback.
local_env = Path(".env")
root_env  = Path("..") / ".env"
load_dotenv(local_env)
load_dotenv(root_env)

print(f"{'✅' if local_env.exists() else '⚠️ '} ./.env                 {'found' if local_env.exists() else 'not found'}")
print(f"{'✅' if root_env.exists()  else '  '} ../.env (repo root)   {'found' if root_env.exists() else 'not found'}")
print()

key = os.environ.get("OPENAI_API_KEY", "")
if not key:
    print("❌ OPENAI_API_KEY is not set.")
    print("   Fix: cp .env.example .env    then edit .env and paste your key")
    print("   Get a key: https://platform.openai.com/api-keys")
elif key == "sk-your-key-here":
    print("❌ OPENAI_API_KEY is still the placeholder from .env.example.")
    print("   Fix: edit .env and paste your real key")
else:
    print(f"✅ OPENAI_API_KEY loaded  (starts {key[:7]}..., {len(key)} chars)")


✅ ./.env                 found
   ../.env (repo root)   not found

✅ OPENAI_API_KEY loaded  (starts gsk_w0a..., 56 chars)


### 0.5 — What the course installer put on your machine

Everything above tested *this notebook*. This cell looks at what the
[course environment installer](https://github.com/AI-Maker-Space/Interactive-Dev-Environment-for-AI-Engineers)
actually created, so you can see the layout rather than trust it.

Everything the installer manages lives in **one directory**, `~/ai-eng`, plus Claude Code
in `~/.local/bin`. That is deliberate: nothing is scattered across your machine, and
removing that one directory undoes almost all of it.

In [5]:
from pathlib import Path

AI_ENG = Path(os.environ.get("AI_ENG_HOME", Path.home() / "ai-eng"))
print(f"Installer home: {AI_ENG}")
print(f"{'✅ found' if AI_ENG.is_dir() else '❌ not found'}\n")

if not AI_ENG.is_dir():
    print("The course installer has not run on this machine.")
    print("Run it — see the commands in the next cell, or this folder's README.md.")
else:
    # What the installer writes, and what each piece is for.
    expected = {
        "env.sh":           "load the tools into a terminal  (Mac/Linux)",
        "env.ps1":          "load the tools into a terminal  (Windows)",
        "setup.sh":         "the installer itself            (Mac/Linux)",
        "setup.ps1":        "the installer itself            (Windows)",
        "README.txt":       "installer notes, rewritten every run",
        "setup.log":        "full log of the last run",
        "setup-report.txt": "written only when something FAILed or WARNed",
        "bin":              "uv lives here",
        "python":           "Python installs (UV_PYTHON_INSTALL_DIR)",
        "nvm":              "nvm, for Node                   (Mac/Linux)",
        "node":             "portable Node 24                (Windows)",
        "path-backup.txt":  "your PATH before it was edited  (Windows)",
    }
    for name, why in expected.items():
        p = AI_ENG / name
        print(f"  {'✅' if p.exists() else '  '} {name:<18} {why}")

    print()
    claude_bin = Path.home() / ".local" / "bin"
    print(f"  {'✅' if claude_bin.is_dir() else '  '} {'~/.local/bin':<18} Claude Code lives here (not in ai-eng)")

    # If the last run flagged anything, it left a report. Surface it.
    report = AI_ENG / "setup-report.txt"
    if report.exists():
        print(f"\n⚠️  A setup report exists, so the last run had a FAIL or WARN:")
        print("─" * 60)
        print(report.read_text(errors="replace")[:1500])
        print("─" * 60)
        print("The installer also copies this to your clipboard — paste it in the help channel.")
    else:
        print("\n✅ No setup-report.txt, which means the last run had no FAIL or WARN.")


Installer home: C:\Users\PAREKH_D\ai-eng
✅ found

     env.sh             load the tools into a terminal  (Mac/Linux)
  ✅ env.ps1            load the tools into a terminal  (Windows)
     setup.sh           the installer itself            (Mac/Linux)
  ✅ setup.ps1          the installer itself            (Windows)
  ✅ README.txt         installer notes, rewritten every run
  ✅ setup.log          full log of the last run
  ✅ setup-report.txt   written only when something FAILed or WARNed
  ✅ bin                uv lives here
  ✅ python             Python installs (UV_PYTHON_INSTALL_DIR)
     nvm                nvm, for Node                   (Mac/Linux)
     node               portable Node 24                (Windows)
     path-backup.txt    your PATH before it was edited  (Windows)

  ✅ ~/.local/bin       Claude Code lives here (not in ai-eng)

⚠️  A setup report exists, so the last run had a FAIL or WARN:
────────────────────────────────────────────────────────────
ï»¿AI Engineer setup

### 0.6 — The installer's own check

Your machine is the installer's business, not this notebook's. When the two disagree,
the installer wins. Re-run it in **check mode**, which changes nothing:

```bash
bash ~/ai-eng/setup.sh --check                                          # Mac/Linux
powershell -ExecutionPolicy Bypass -File $HOME\ai-eng\setup.ps1 -Check   # Windows
```

You want **All good**. It exits non-zero if anything `[FAIL]`ed, so it also works in a script.

**Reading its output:**

| Marker | Means | Do |
|---|---|---|
| `[ OK ]` | already installed and working | nothing |
| `[DONE]` | installed or fixed on this run | nothing |
| `[ -- ]` | optional, skipped | nothing — this is fine |
| `[WARN]` | works, but something is worth a look | read the `->` hint under it |
| `[FAIL]` | needs fixing | read the `->` hint under it |

**Not installed yet?** The one-click install, from the course's
[installer repo](https://github.com/AI-Maker-Space/Interactive-Dev-Environment-for-AI-Engineers):

```bash
# Mac/Linux
mkdir -p ~/ai-eng && curl -fsSL https://raw.githubusercontent.com/AI-Maker-Space/Interactive-Dev-Environment-for-AI-Engineers/main/setup.sh -o ~/ai-eng/setup.sh && bash ~/ai-eng/setup.sh
```

```powershell
# Windows
mkdir -Force $HOME\ai-eng | Out-Null; irm https://raw.githubusercontent.com/AI-Maker-Space/Interactive-Dev-Environment-for-AI-Engineers/main/setup.ps1 -OutFile $HOME\ai-eng\setup.ps1; powershell -ExecutionPolicy Bypass -File $HOME\ai-eng\setup.ps1
```

It is **safe to re-run at any time** — anything already done is reported `[ OK ]` and
skipped. Useful flags:

| Mac/Linux | Windows | What it does |
|---|---|---|
| `--check` | `-Check` | report only, change nothing |
| `--tools-only` | `-ToolsOnly` | skip the Git name/email and GitHub SSH steps |
| `--yes` / `-y` | `-Yes` | ask nothing, take every default |
| `--no-copy` | `-NoCopy` | don't copy the report to your clipboard |
| `--help` / `-h` | — | print usage and exit |

Full setup walkthrough, including the VS Code install that comes first:
[Initial Setup](../00_Docs/Prerequisites/Initial_Setup/README.md) ·
[step-by-step DIY guide](https://github.com/AI-Maker-Space/Interactive-Dev-Environment-for-AI-Engineers/blob/main/details/README.md)

---

<a id="part-1"></a>
# Part 1 — Version control

<a id="why-version-control"></a>
## Why version control

Not "because you might lose your work." Backups solve that. Version control solves a
different problem: **more than one version of the truth has to exist at the same time.**

Picture one environment. You have an app that works, and you want to change it. The
moment you start editing, the working version is gone — you are editing the only copy
users have. Every change is a bet you cannot take back.

So you split the timeline:

| Stage | What it is | Who sees it | What is allowed to be true here |
|---|---|---|---|
| **dev** | your machine | you | broken, half-finished, experimental |
| **production** | what users use | everyone | only what works |

That is the two-stage model, and it is already enough to stop you breaking production
by typing. But it has a gap: the jump from "works on my laptop" to "live for everyone"
is one move, untested anywhere real.

So teams add a third:

| Stage | What it is | Who sees it | Why it exists |
|---|---|---|---|
| **dev** | your machine | you | write code |
| **staging** | a real deployment, fake users | the team | *this is where you find out your change was wrong* |
| **production** | the real thing | everyone | only what survived staging |

**Staging is the whole argument for version control.** If code can live in three states
at once, you need something that can hold three states at once and move changes between
them deliberately. That is git. The branches are not bureaucracy; they are the three
stages made concrete:

| Branch | Maps to | In this course |
|---|---|---|
| `main` | production. "The current official story" | where your submitted work lives |
| `develop` | staging. "The draft of the next chapter" | not used — we keep it simple |
| `feature/*` | dev. "Paragraphs before they go in the book" | use freely for experiments |

> **For this course we work directly on `main`.** That is a deliberate simplification,
> and it is the right call while you are learning. Know that the real version exists,
> and that the reason is staging. More: [Version Control](../00_Docs/Prerequisites/Initial_Setup/3_Version_Control/README.md).

<a id="how-git"></a>
## How version control — this repository is a git

Abstract diagrams of git do not stick. The repository you have open **is** a git, so
let's read it. Every cell below runs a real command against your actual clone.

### The three places your code lives

| Name | Where | What you do with it |
|---|---|---|
| **upstream** | AI Makerspace's repo | `pull` from it. Never push |
| **origin** | *your* fork on GitHub | `push` to it. This is what you submit |
| **local** | the folder on your laptop | edit files here |

Flow: `upstream` → `origin` → `local` → `origin`.

In [6]:
import subprocess
from pathlib import Path

REPO_ROOT = Path("..").resolve()

def git(*args, cwd=REPO_ROOT):
    """Run a git command and return its output as text (stdout+stderr)."""
    out = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    return (out.stdout + out.stderr).strip()

print(f"Repo root: {REPO_ROOT}")
print()
print("── Your remotes (git remote -v) ──")
print(git("remote", "-v") or "(no remotes configured)")


Repo root: C:\Projects\AIE11\AIE11

── Your remotes (git remote -v) ──
origin	https://github.com/dp6416/AIE11.git (fetch)
origin	https://github.com/dp6416/AIE11.git (push)
upstream	https://github.com/AI-Maker-Space/AIE11.git (fetch)
upstream	DISABLED (push)


**Reading that output.** You should see `origin` twice (fetch and push). If you also see
`upstream`, your fork is wired up for course updates.

**No `upstream`?** That is the one-time setup you only do once per course:

```bash
git remote add upstream https://github.com/AI-Maker-Space/AIE11.git
git remote -v    # confirm it took
```

Full walkthrough: [Set Up Initial Repos](../00_Docs/Prerequisites/Initial_Setup/2_Set_Up_Initial_Repos/README.md).


In [7]:
print("── Where you are (git status -sb) ──")
print(git("status", "-sb"))
print()
print("── Recent history (git log --oneline -8) ──")
print(git("log", "--oneline", "-8"))


── Where you are (git status -sb) ──
## main...origin/main
MM 01_Life_As_An_AI_Engineer/Life_As_An_AI_Engineer.ipynb
?? 01_Life_As_An_AI_Engineer/check_openai_connection.py

── Recent history (git log --oneline -8) ──
8b674ae testing git commands
347ee8b test commit message
e5de9b4 First commit - ran ipynb file and added some content to it.
51436e2 Session 1 - Quicklinks
bf59e6a Session 1 - Quicklinks
986cc1e Update README.md
98c7631 AIE11 - Week 1 - Session 1
78fb70d Update README.md


**Reading that output.**

- The first line of `status -sb` is your branch, and `[ahead N]` / `[behind N]` tells you
  how far local has drifted from origin. `ahead 2` means two commits you have not pushed.
- `??` marks files git has never seen. `M` marks tracked files you changed.
- Each `git log` line is one commit: a snapshot you can return to. That is the unit of
  work version control actually deals in.

### The commit you will make every week

A commit is three commands, and the middle one is the one people skip:

```bash
git add .                       # stage: choose what goes in
git diff --staged               # READ IT. this is the step that saves you
git commit -m "message"         # snapshot it
git push origin main            # send it to your fork
```

> ⚠️ **`git diff --staged` before every commit.** This is how you catch a `.env` with a
> live API key before it becomes permanent. A key pushed to a public repo is scraped in
> minutes, and deleting the commit does not help — you have to rotate the key.

Let's prove your `.env` is actually protected:


In [8]:
# Is .env ignored? Ask git, don't assume.
ignored = git("check-ignore", "-v", "01_Life_As_An_AI_Engineer/.env")
if ignored:
    print(f"✅ .env is gitignored by rule:\n   {ignored}")
else:
    print("⚠️  git does NOT report .env as ignored.")
    print("   Check .gitignore before you ever commit, or your key goes public.")

print()
print("── Anything currently staged that shouldn't be? ──")
staged = git("diff", "--staged", "--name-only")
print(staged if staged else "(nothing staged — clean)")


✅ .env is gitignored by rule:
   .gitignore:4:.env*	01_Life_As_An_AI_Engineer/.env

── Anything currently staged that shouldn't be? ──
01_Life_As_An_AI_Engineer/Life_As_An_AI_Engineer.ipynb


### Your weekly loop

```bash
git checkout main
git pull upstream main      # get new course material
git push origin main        # keep your fork level with it
# ... do the assignment ...
git add . && git diff --staged && git commit -m "week N" && git push origin main
```

That is the whole routine for ten weeks. Detail:
[Your Weekly Git Workflow](../00_Docs/Prerequisites/Initial_Setup/4_Your_Weekly_Git_Workflow/README.md).


---

<a id="part-2"></a>
# Part 2 — The LLM API call

<a id="why-api"></a>
## Why call an API at all?

You have used ChatGPT. Why not just use that?

Because a chat window is a **product**, and you are here to build products. The
difference is not the model — it is the same model. The difference is what you can do
with the call:

| In the chat window | Through the API |
|---|---|
| You type, you read | *Your code* types, and your code reads |
| One conversation | Ten thousand conversations, in parallel |
| You are the one deciding what to send | You decide the rules **before** the user arrives |
| Output is text on a screen | Output is a typed object your program acts on |

That third row is the real one. As a user you write a message. As an engineer you write
the **system instructions** — the rules that apply to every user, forever — and the
user's message arrives inside the frame you built. That asymmetry is the job.

> The same principle is why `AGENTS.md` in this repo can steer Claude Code into teaching
> mode. System-level instructions outrank what comes after them. You have already been
> on the engineering side of this without noticing.

<a id="three-roles"></a>
## The three roles

As a *user* of an LLM app, the shape is: text in, text out.

As a *developer*, it is: **a list of role-tagged messages in, one message out.**

| Role | Also called | Who writes it | What it does |
|---|---|---|---|
| `system` | developer | **you, the engineer** | the rules. Prioritised above everything else |
| `user` | human | your app's user (or you, faking one) | the request |
| `assistant` | AI | the model (or you, faking one) | a reply — and you can **put words in its mouth** |

That last one looks strange and is the key to few-shot prompting in Part 3: you write
fake `assistant` turns to demonstrate the behaviour you want, and the model continues
the pattern.

More: [LLM APIs — The "System" Role](../00_Docs/Prerequisites/Initial_Setup/6_LLM_APIs/README.md).

<a id="api-setup"></a>
## Setup — the client and two helpers

We use the OpenAI SDK against the **Responses API** (`client.responses.*`), the modern
surface for gpt-5-era models. This is the same helper layer as
[`instructor_materials/Prompt_Engineering_Patterns.ipynb`](instructor_materials/Prompt_Engineering_Patterns.ipynb) in this folder,
so the patterns you learn here carry straight over.

Override the model or endpoint in `.env` without touching code:

```
LLM_MODEL=gpt-5.4-mini                   # any OpenAI-compatible model string
LLM_API_BASE=http://localhost:4000/v1    # a LiteLLM proxy, a local server, ...
```

In [9]:
import os, time, json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv(Path(".env"))          # this folder's .env (preferred)
load_dotenv(Path("..") / ".env")   # ...or the repo root's
assert os.environ.get("OPENAI_API_KEY"), \
    "No OPENAI_API_KEY. Run `cp .env.example .env` and add your key, then restart the kernel."

MODEL    = os.getenv("LLM_MODEL", "gpt-5.4-mini")
API_BASE = os.getenv("LLM_API_BASE") or None
client   = OpenAI(base_url=API_BASE)

# ── Thin helpers ────────────────────────────────────────────────────────────

def ask(input, instructions=None, effort=None, **kw):
    """
    One call against the Responses API.
    `input`        — a plain string, or a list of role-message dicts
    `instructions` — the system / developer instruction string
    `effort`       — reasoning effort: None | 'low' | 'medium' | 'high'
                     (gpt-5+ models only; None = no explicit reasoning step)
    """
    params = {"model": MODEL, "input": input}
    if instructions is not None:
        params["instructions"] = instructions
    if effort is not None:
        params["reasoning"] = {"effort": effort}
    return client.responses.create(**params, **kw)

def show(resp) -> None:
    """Pretty-print a Response object's text into the notebook."""
    display(Markdown(resp.output_text))

print(f"✅ Model: {MODEL}   API base: {API_BASE or 'OpenAI default'}")


✅ Model: openai/gpt-oss-120b   API base: https://api.groq.com/openai/v1


### Your first call

One string in, one answer out. This is the whole API in its simplest form.


In [10]:
resp = ask("In two sentences, what does an AI engineer do that a data scientist doesn't?")
show(resp)


An AI engineer focuses on designing, building, and deploying scalable machine‑learning models and systems, handling software engineering tasks such as model integration, real‑time inference pipelines, and performance optimization. In contrast, a data scientist primarily concentrates on exploratory analysis, statistical modeling, and deriving insights from data, without necessarily turning those models into production‑ready applications.

### What came back besides the text

The text is what you read; the **metadata** is what you engineer against. Token counts
are what you pay for, and latency is what your user waits through. Look at both now, in
week 1, so the numbers in week 4 are not a surprise.


In [11]:
t0 = time.perf_counter()
resp = ask("Name the three message roles in an LLM API call. Just the three words.")
elapsed = time.perf_counter() - t0

print("── text ──")
print(resp.output_text)
print()
print("── metadata ──")
print(f"model          : {resp.model}")
print(f"latency        : {elapsed:.2f}s")
print(f"input tokens   : {resp.usage.input_tokens}")
print(f"output tokens  : {resp.usage.output_tokens}")
print(f"total tokens   : {resp.usage.total_tokens}")


── text ──
system, user, assistant

── metadata ──
model          : openai/gpt-oss-120b
latency        : 0.45s
input tokens   : 88
output tokens  : 56
total tokens   : 144


### The three roles, for real

Now the developer-side shape: a **list** of role-tagged messages. Note the `assistant`
turn we wrote ourselves — the model has no idea we faked it, and it continues the
clipped style we demonstrated.


In [12]:
conversation = [
    {"role": "user",      "content": "What is uv?"},
    {"role": "assistant", "content": "A fast Python package and version manager. Replaces pip, venv, and pyenv."},
    {"role": "user",      "content": "And what is git?"},
]

print("The assistant turn above is OURS — we wrote it to set a terse style.")
print("Watch the answer below match that style without being asked:")
print("=" * 60)
show(ask(conversation))


The assistant turn above is OURS — we wrote it to set a terse style.
Watch the answer below match that style without being asked:


**Git** is a free, open‑source **distributed version‑control system (DVCS)** designed to track changes in source code (and, more generally, any set of files) over time. It was created by Linus Torvalds in 2005 to help manage the development of the Linux kernel, and it has since become the de‑facto standard tool for collaborative software development.

---

## Core ideas

| Concept | What it means | Why it matters |
|---------|---------------|----------------|
| **Repository (repo)** | A directory that Git manages, containing your project files and a hidden `.git` folder with all the history and metadata. | All version‑control data lives here; you can clone, pull, push, etc. |
| **Commit** | A snapshot of the entire project at a given point, together with a message, author info, and a unique SHA‑1 (or SHA‑256) hash. | Provides an immutable record you can return to, compare, or share. |
| **Branch** | A lightweight, movable pointer to a commit. Each branch represents an independent line of development. | Enables parallel work (feature branches, bug‑fix branches) without interfering with the main code line. |
| **Merge / Rebase** | Operations that combine changes from one branch into another. | Merge preserves history; rebase rewrites it to produce a linear history. |
| **Remote** | A version of the repo hosted elsewhere (e.g., on GitHub, GitLab, Bitbucket). | Allows multiple developers to share changes over a network. |
| **Distributed** | Every clone contains the full history, not just a snapshot. | You can work offline, recover from server failures, and have many independent “central” copies. |

---

## Typical workflow (simplified)

1. **Clone** a remote repo: `git clone https://github.com/user/project.git`
2. **Create a branch** for your work: `git checkout -b my‑feature`
3. **Edit files** and **stage** changes: `git add file1.py file2.py`
4. **Commit** the changes: `git commit -m "Add new feature X"`
5. **Pull** any new work from the remote (to avoid conflicts): `git pull --rebase origin main`
6. **Push** your branch to the remote: `git push origin my‑feature`
7. **Open a Pull Request / Merge Request** on the hosting platform for review.
8. After review, **merge** the branch into the main line (often done via the platform’s UI).

---

## Why developers love Git

- **Speed** – Most operations are local and therefore very fast.
- **Safety** – Commits are immutable; you can always revert or reset.
- **Flexibility** – Supports many branching strategies (Git Flow, GitHub Flow, trunk‑based development, etc.).
- **Collaboration** – Works seamlessly with hosting services (GitHub, GitLab, Bitbucket) that add issue tracking, CI/CD pipelines, code review, and more.
- **Extensibility** – Hooks, custom commands, and integrations let you tailor Git to your workflow.

---

## Common commands (cheat‑sheet)

| Command | Purpose |
|---------|---------|
| `git init` | Create a new local repository. |
| `git clone <url>` | Copy an existing repo (including its history). |
| `git status` | Show staged, unstaged, and untracked files. |
| `git add <path>` | Stage changes for the next commit. |
| `git commit -m "msg"` | Record a new snapshot. |
| `git log` | View commit history. |
| `git branch` | List branches; `git branch <name>` creates one. |
| `git checkout <branch>` | Switch to another branch. |
| `git merge <branch>` | Merge another branch into the current one. |
| `git rebase <branch>` | Replay current branch’s commits onto another branch. |
| `git pull` | Fetch from remote and merge (or rebase). |
| `git push` | Send local commits to a remote. |
| `git reset --hard <commit>` | Move the current branch to a specific commit, discarding later changes. |
| `git stash` | Temporarily save uncommitted changes. |
| `git diff` | Show differences between commits, branches, or the working tree. |

---

## Resources to learn more

- **Pro Git (free book)** – https://git-scm.com/book/en/v2  
- **Interactive tutorial** – https://learngitbranching.js.org/  
- **Official documentation** – https://git-scm.com/doc  

Git is a powerful tool, but you don’t need to master every nuance to be productive. Start by learning the basic cycle (clone → edit → add → commit → push) and gradually explore branches, merges, and rebases as your workflow demands. Happy coding!

---

<a id="part-3"></a>
# Part 3 — Prompt engineering, five patterns

> These are condensed from [`instructor_materials/Prompt_Engineering_Patterns.ipynb`](instructor_materials/Prompt_Engineering_Patterns.ipynb)
> in this folder, which has seven patterns and more depth on each. Work through that one
> next — this is the orientation pass.

Each pattern below is shown **weak first, then strong**. The weak version is not a straw
man; it is what you will write by default. Seeing the two side by side is the point.

<a id="pattern-1"></a>
## Pattern 1 — Persona: instructions steer voice

The `instructions` parameter is your system prompt. It does not change what the model
**knows**. It changes how the model **speaks**. Same question, same facts, different voice.

This is the first thing to reach for on anything user-facing: who should this sound like?

In [13]:
Q = "A teammate committed an API key to our public repo. What do we do?"

print("=" * 60)
print("WEAK — no persona (whatever the default voice is)")
print("=" * 60)
show(ask(Q))


WEAK — no persona (whatever the default voice is)


### TL;DR  
1. **Revoke / rotate the key immediately** (or create a new one and delete the old).  
2. **Remove the secret from the Git history** (rewrite, push, force‑push).  
3. **Invalidate any tokens that might have been generated from that key**.  
4. **Audit logs for misuse** and alert any affected parties.  
5. **Patch the workflow** so it can’t happen again (git‑hooks, secret‑scanning, .gitignore, CI checks, training).

Below is a step‑by‑step playbook you can run through with your team.

---

## 1. Stop the key from being usable

| Action | Why | How |
|--------|-----|------|
| **Revoke / rotate the API key** | The moment the key is public anyone can call the service. Revoke it before an attacker can exploit it. | • Log into the service’s console (AWS, GCP, Stripe, SendGrid, etc.). <br>• Delete the exposed key or generate a new one. <br>• Update any internal services/config that used the old key. |
| **Create a short‑lived replacement** (if you can’t immediately update all callers) | Gives you a “window” to roll out the new secret safely. | Generate a new key, store it securely, and push it to the places that need it (environment variables, secret manager, etc.). |
| **Invalidate derived credentials** (e.g., JWT signing keys, OAuth tokens) | Some services issue short‑lived tokens that were signed with the compromised secret. | Revoke all active sessions, force a sign‑out, or rotate the signing secret. |

**Do it now** – before you start cleaning the repo.

---

## 2. Erase the secret from the repository history

### A. Identify every commit that contains the key
```bash
git rev-list --all | while read rev; do
  git grep -F 'YOUR_PART_OF_THE_KEY' $rev && echo "found in $rev"
done
```
*If you use a secret‑scanning tool (GitGuardian, TruffleHog, Gitleaks), run that too – it will locate hidden occurrences.*

### B. Rewrite history

#### Option 1 – BFG Repo‑Cleaner (simpler, fast)
```bash
# 1️⃣ Install BFG (java -jar bfg.jar)
# 2️⃣ Create a file with the exact secret (or a pattern)
echo 'YOUR_API_KEY' > keys.txt

# 3️⃣ Run BFG
java -jar bfg.jar --replace-text keys.txt my-repo.git

# 4️⃣ Clean up refs & gc
cd my-repo.git
git reflog expire --expire=now --all && git gc --prune=now --aggressive
```

#### Option 2 – Native `git filter-repo` (more control)
```bash
# Install: pip install git-filter-repo
git filter-repo --replace-text <(printf 'YOUR_API_KEY==REDACTED\n')
```

### C. Force‑push the cleaned history
```bash
git push --force --all
git push --force --tags
```

> **Important:** All collaborators must **re‑clone** (or reset their local clones) because the history has changed. Communicate the required steps clearly:
```bash
git fetch origin
git reset --hard origin/main   # or the default branch
```

### D. Remove cached copies on hosting services
- GitHub: go to **Settings → Code security and analysis → Secret scanning** → “Dismiss” the alert after you’ve rotated the key.
- GitLab / Bitbucket: similar “secret detection” pages.
- In CI/CD caches (e.g., GitHub Actions caches, CircleCI workspaces), delete any caches that might still hold the old repo snapshot.

---

## 3. Check for misuse (log analysis)

1. **Enable/collect logs** from the service that the key accesses. Look for:
   - Unexpected IP addresses
   - Unusual request patterns (high volume, odd endpoints)
   - Calls made outside of normal business hours
2. **Set up alerts** for the next 48‑72 h (most attackers act quickly).
3. **If you see suspicious activity**, treat it as a breach:
   - Rotate any downstream secrets.
   - Notify customers or partners if data was accessed.
   - Follow your organization’s incident‑response plan (legal, compliance, PR).

---

## 4. Communicate internally & externally

| Audience | Message |
|----------|---------|
| **Team / Devs** | “API key X was exposed and has been revoked. Please pull the latest repository (cleaned history) and update your local env vars.” |
| **Management / Security Lead** | Brief incident summary, impact assessment, timeline of mitigation steps. |
| **External provider** (e.g., Stripe, Twilio) | Some providers have a “compromised key” form – let them know in case they see anomalous traffic. |
| **Customers / Users** (if data could be compromised) | Follow your legal/compliance policy (GDPR, CCPA, etc.) – you may need a breach notification. |

---

## 5. Harden the workflow to prevent future leaks

| Technique | How to implement |
|-----------|-------------------|
| **Never store secrets in code** | Use environment variables, secret‑management services (AWS Secrets Manager, HashiCorp Vault, GCP Secret Manager). |
| **Add secrets to `.gitignore`** | `*.key`, `*.pem`, `config/*.env`, etc. |
| **Git hooks** (client‑side) | `pre-commit` hook that runs `git grep -I -n $SECRET_PATTERN && exit 1` – aborts if a secret is found. |
| **CI secret‑scanning** | Enable built‑in scanners (GitHub Advanced Security, GitLab Secret Detection, Bitbucket Pipelines) **and** run a third‑party scanner (Gitleaks, TruffleHog) on every PR. |
| **Branch protection** | Require PR review, disallow force‑pushes on main, enforce status checks. |
| **Education** | Short “security hygiene” session: why API keys are like passwords, how to use `.env.example` templates, etc. |
| **Rotate regularly** | Set a policy (e.g., every 90 days) for rotating high‑privilege keys. |
| **Use short‑lived tokens** | Where possible, use OAuth or signed JWTs that expire in minutes/hours rather than static API keys. |
| **Audit tool** | Run `git log -S 'YOUR_PART_OF_THE_KEY'` regularly or schedule a nightly scan. |

---

## 6. Checklist you can copy‑paste for the next incident

```
[ ] 1. Revoke/rotate the exposed key on the service console.
[ ] 2. Identify every commit containing the secret.
[ ] 3. Rewrite repo history (BFG / filter-repo) and force‑push.
[ ] 4. Instruct all collaborators to re‑clone or reset.
[ ] 5. Delete any cached copies on CI/CD, GitHub, etc.
[ ] 6. Scan logs for unauthorized usage → document findings.
[ ] 7. Notify internal stakeholders & external provider if needed.
[ ] 8. Draft breach‑notification (if data may be compromised).
[ ] 9. Add/strengthen pre‑commit hook and CI secret‑scan.
[ ]10. Update documentation: .gitignore, secret‑management guide.
[ ]11. Schedule a short post‑mortem meeting (what happened, why, how to avoid).
```

---

### Quick “one‑liner” for a busy dev:

> *“We revoked the key, rewrote the repo with BFG, forced a push, and told everyone to pull the cleaned history. I’ve added a pre‑commit secret‑scan and enabled GitHub’s secret‑detection. Let’s monitor the provider logs for the next 48 h for any odd activity.”*

---

## 7. Resources (free & open‑source)

| Tool | Primary use | Link |
|------|-------------|------|
| **BFG Repo‑Cleaner** | Fast mass removal of passwords/keys | https://rtyley.github.io/bfg-repo-cleaner/ |
| **git-filter-repo** | Official replacement for `filter-branch` | https://github.com/newren/git-filter-repo |
| **Gitleaks** | Scans repo history & PRs for secrets | https://github.com/gitleaks/gitleaks |
| **TruffleHog** | Finds high‑entropy strings, regex‑based secrets | https://github.com/trufflesecurity/trufflehog |
| **GitGuardian (free tier)** | SaaS secret‑detection for GitHub/GitLab/Bitbucket | https://www.gitguardian.com/ |
| **GitHub Advanced Security** | Built‑in secret scanning & codeQL | https://github.com/features/security |
| **pre‑commit** framework | Manage hooks like `detect-secrets` | https://pre-commit.com/ |

---

### Bottom line
1. **Revoke** the key now.  
2. **Erase** it from every commit (history rewrite).  
3. **Audit** for misuse.  
4. **Communicate** and **document**.  
5. **Harden** your workflow to make sure the next secret never lands in the repo.

If you follow those steps, you’ll contain the exposure, reduce the chance of any damage, and put safeguards in place for the future. Good luck, and feel free to ask if you need help with a specific tool or policy draft!

In [14]:
TERSE_SRE = (
    "You are a terse senior site-reliability engineer during an incident. "
    "Give numbered, imperative steps. No preamble, no reassurance, no caveats."
)

PATIENT_MENTOR = (
    "You are a patient mentor teaching a junior engineer. "
    "Explain why each step matters, anticipate the follow-up question, "
    "and end with one thing to remember next time."
)

for label, persona in [("Persona A: terse SRE", TERSE_SRE),
                       ("Persona B: patient mentor", PATIENT_MENTOR)]:
    print("=" * 60)
    print(f"STRONG — {label}")
    print("=" * 60)
    show(ask(Q, instructions=persona))
    print()


STRONG — Persona A: terse SRE


1. Revoke the exposed API key immediately.  
2. Generate a new API key with the same permissions.  
3. Update all services, CI/CD pipelines, and local environments to use the new key.  
4. Remove the key from the repository history using `git filter-repo` or `bfg --delete-files`.  
5. Force‑push the cleaned history and invalidate any cached clones.  
6. Add the key to the `.gitignore` and store it in a secret manager or vault.  
7. Scan the entire codebase and CI logs for other accidental exposures.  
8. Enable automated secret scanning on the repository.  
9. Rotate any dependent secrets that may have been compromised.  
10. Document the incident, update the incident response runbook, and communicate the remediation to stakeholders.


STRONG — Persona B: patient mentor


### Quick‑fire Action Plan  
| Step | What you do | Why it matters |
|------|-------------|----------------|
| **1️⃣ Revoke / rotate the key immediately** | Log in to the service that issued the API key and either **revoke** it or **generate a new one**. Update any services that were using the old key. | As soon as the key is exposed anyone on the internet can call the API. Revoking stops any unauthorized use right away. |
| **2️⃣ Remove the key from the Git history** | ```bash\n# make a backup first\ncp -a .git .git.backup\n# Use the BFG or git filter‑repo to purge the secret\nbfg --delete-files <filename‑or‑pattern>\n# or, for a single line:\ngit filter-repo --replace-text <(printf 'YOUR_OLD_KEY==>REDACTED')\n# Force‑push the cleaned history\ngit push --force --all\n``` | Deleting the key from the latest commit isn’t enough—Git stores every previous snapshot. If the secret remains in history, anyone can still fetch it. |
| **3️⃣ Invalidate any cached clones** | Ask every developer, CI system, and any third‑party that might have cloned the repo to **delete** their local copy and re‑clone the cleaned repository. | Even after you rewrite history, old clones still contain the secret. A fresh clone guarantees no stray copies remain. |
| **4️⃣ Scan for other secrets** | Run a repository‑wide scan with tools such as **Git‑Guardian, truffleHog, or gitleaks**. Add the scan to CI so it runs on every PR. | The first leak often uncovers more. Automated scanning catches anything you missed manually. |
| **5️⃣ Harden the workflow** | • Store secrets in environment variables, secret‑management services (AWS Secrets Manager, Vault, etc.)<br>• Add a **`.gitignore`** entry for files that contain keys (e.g., `*.env`)<br>• Enable **pre‑commit hooks** that run a secret‑detection script<br>• Turn on **branch protection** so only reviewed PRs can be merged | Prevents the same mistake from happening again. The more checks you have before code reaches the repo, the fewer secrets slip through. |
| **6️⃣ Communicate** | Send a short incident note to the team: what happened, what you did (revoked key, cleaned history), and what to do next (pull fresh repo, update configs). | Transparency keeps trust high and ensures everyone updates their local copies and credentials. |

---

### Why each step matters (the “mentor” explanation)

1. **Revoke/rotate first** – Think of the key as a physical password written on a sticky note that you just stuck on a billboard. The moment anyone can see it, you need to lock the door (revoke) before you start cleaning up the billboard. If you wait, an attacker could already be hammering your API, racking up charges or stealing data.

2. **Rewrite history** – Git isn’t a single snapshot; it’s a chain of snapshots. Deleting the line from the most recent commit is like erasing the word from the front page of a newspaper but leaving the old editions on the newsstand. Tools like **BFG** or **git‑filter‑repo** rewrite every commit that ever contained the secret, making the secret disappear from the repo’s permanent record.

3. **Force everyone to re‑clone** – Even after you rewrite history, any clone that existed before the rewrite still has the old commits. If a developer runs `git pull`, Git will try to merge the old (still‑containing‑secret) history with the new cleaned one, re‑introducing the secret. A fresh clone guarantees a clean slate.

4. **Run a scanner** – Human eyes often miss a stray key in a config file or a comment. Automated scanners look for patterns that resemble keys (long random strings, JWTs, etc.) and can flag them before they ever get merged. Adding the scanner to CI makes the check repeatable and unavoidable.

5. **Hardening** – Storing secrets in code is a recipe for future accidents. By moving them out of the repo and into a managed secret store, you treat them like any other credential: you fetch them at runtime, you audit who can read them, and you can rotate them without touching source code. Hooks and `.gitignore` act as safety nets that catch a slip before it lands in the repository.

6. **Communicate** – A security incident is a team event, not just an individual mistake. If you don’t tell people to pull the fresh repo and replace the key, you’ll end up with a few “ghost” copies still leaking. A concise note also reinforces the learning moment without placing blame.

---

### Anticipated follow‑up question  

**“How exactly do I purge the secret from *all* the old commits without breaking the repo?”**  

- Use a purpose‑built tool (BFG or `git filter-repo`) rather than hand‑crafting `git filter-branch`.  
- Example with **BFG** (the simplest for a single secret):  

  ```bash
  # 1️⃣ Install BFG (Java required)
  brew install bfg   # macOS, or download the .jar

  # 2️⃣ Run it, pointing at the exact string or a file pattern
  bfg --replace-text <(printf 'MY_SUPER_SECRET==>REDACTED')  .
  ```

- After the rewrite, verify with `git log -p` that the key is gone, then **force‑push** (`git push --force --all`).  
- Remember: any fork or mirror of the repo will still have the secret until its owner also rewrites or deletes their copy, so you may need to contact external collaborators.

---

### One thing to remember next time  

> **Never let a secret live in the repository’s history; treat it like a password you’d never write on a sticky note.**  

If you keep secrets out of the repo from day 1 (environment variables, secret‑manager, `.gitignore`), you eliminate the whole class of “accidental‑commit” incidents. 🌟

Same model, same facts, two completely different deliverables. **The instructions are the
character brief.** In production these live in versioned prompt templates, so every team
ships the same model with one locked brand voice.


<a id="pattern-2"></a>
## Pattern 2 — Few-shot: teaching a pattern by example

The model already knows what words mean. Few-shot teaches it a **task-specific
convention** it could not possibly guess — with no training, just examples in the prompt.

The cleanest proof uses made-up words (borrowed from the
[GPT-3 paper](https://arxiv.org/abs/2005.14165)). The model has never seen "stimple" or
"falbean", so whatever it does with them came from your examples alone.

In [15]:
print("=" * 60)
print("WEAK — zero-shot: it has to guess what these words mean")
print("=" * 60)
show(ask("Use the words 'stimple' and 'falbean' in a sentence."))


WEAK — zero-shot: it has to guess what these words mean


The botanist examined the rare falbean orchid, noting how its stimple-shaped petals curled delicately around the stem.

In [16]:
# Teach the meanings as prior turns — the fake-assistant trick from Part 2.
few_shot = [
    {"role": "user",      "content": "Something 'stimple' is good, high-quality, and well-functioning. "
                                     "Use 'stimple' in a sentence."},
    {"role": "assistant", "content": "'Boy, that there is a stimple drill.'"},
    {"role": "user",      "content": "A 'falbean' is a tool used to fasten, tighten, or spin things. "
                                     "Use both 'stimple' and 'falbean' in a sentence."},
]

print("=" * 60)
print("STRONG — few-shot: definitions supplied as prior turns")
print("=" * 60)
show(ask(few_shot))


STRONG — few-shot: definitions supplied as prior turns


The stimple design of the new smartwatch makes it feel both sleek and reliable, while the built‑in falbean lets you quickly tighten the strap without any extra tools.

That is **in-context learning**: a new convention acquired at call time, no retraining.

The real use is not whimsical words — it is *your* conventions. Which tickets count as
P0 at your company. Which requirements are in scope for this engagement. Which support
emails are refund requests. None of that is inferable from English; all of it is
teachable in two examples.


<a id="pattern-3"></a>
## Pattern 3 — Chain of thought and the reasoning budget

Chain of thought asks the model to **reason before answering**. gpt-5-era models expose
this as a dial: `effort='low'` buys a quick think, `'high'` buys a longer one. More
effort costs more tokens and more seconds — a tradeoff you can measure, below.

The classic test is a problem with a **trap**: a surface detail that tempts a fast,
wrong answer.

In [17]:
TRAP = "I need to get my car cleaned. The car wash is only 50 meters away. Should I walk or drive?"

def timed_ask(prompt, effort):
    t0 = time.perf_counter()
    resp = ask(prompt, effort=effort)
    elapsed = time.perf_counter() - t0
    details = getattr(resp.usage, "output_tokens_details", None)
    reasoning = getattr(details, "reasoning_tokens", "n/a") if details else "n/a"
    return resp, elapsed, reasoning

for label, effort in [("None (no explicit reasoning step)", None), ("'high'", "high")]:
    resp, elapsed, reasoning = timed_ask(TRAP, effort)
    print("=" * 60)
    print(f"effort = {label}")
    print(f"   {elapsed:.2f}s | reasoning tokens: {reasoning} | output: {resp.usage.output_tokens}")
    print("=" * 60)
    show(resp)
    print()


effort = None (no explicit reasoning step)
   2.71s | reasoning tokens: 98 | output: 1126


**Short answer:**  
Walk → it’s the quickest, cheapest, and least wear‑and‑tear option for a 50‑meter trip.  

---

## Why walking usually wins for a 50 m (≈ 0.03 mi) distance

| Factor | Walking | Driving |
|--------|----------|----------|
| **Time** | ~30 seconds (plus a few seconds to get out of the car) | ~10 seconds to start the engine, shift, drive, then park again |
| **Fuel/energy** | 0 gal (no emissions) | ~0.001 gal of gasoline ≈ 0.04 L (tiny, but still a burn) |
| **Vehicle wear** | None | Brake, clutch/torque‑converter, transmission, steering, tires all get a tiny extra cycle |
| **Safety** | Minimal risk (just a short sidewalk or driveway) | Slight risk of hitting curbs, other cars, or pedestrians in a tight space |
| **Convenience** | You already have your keys, no need to start the engine | You must unlock, start, and later re‑park |
| **Weather** | May be a factor if it’s raining, snowing, or extremely hot | Same exposure, but you stay inside the car (which could be hotter or colder) |
| **Physical effort** | Negligible (a few steps) | None, but you’ll have to sit in the car for a few extra minutes while the wash runs |

### Bottom line
- **If the weather is decent (no heavy rain, snow, or extreme heat)**: walk.
- **If it’s pouring, icy, or you’re carrying heavy items**: hop in the car, but still consider pulling up right next to the wash so you can step out quickly.

---

## Quick decision checklist

1. **Weather** – Is it safe and comfortable to walk?  
   - ☀️ Sunny/overcast → walk.  
   - 🌧️ Rain/snow → drive (or wear appropriate rain gear if you still want to walk).

2. **Time pressure** – Do you need the car cleaned right now?  
   - If you’re in a hurry, the extra ~15 seconds saved by driving is negligible; you’ll still spend a few minutes waiting for the wash cycle anyway.

3. **Parking situation** – Is there a spot right by the wash?  
   - If you can park right next to it, driving is just as easy as walking.  
   - If you’d have to maneuver around other cars, walking avoids that hassle.

4. **Physical condition** – Any mobility issues?  
   - If you’re fine walking a few steps, go for it.  
   - If you have a limp, injury, or need a stroller, drive.

5. **Environmental mindset** – Want to minimize emissions?  
   - Walking saves a tiny but real amount of fuel and reduces wear on your car.

---

## Suggested routine (walk‑first approach)

1. **Grab your keys** (or a small bag of any items you need to bring with you).  
2. **Step out** onto the sidewalk or driveway—just ~5–6 steps.  
3. **Walk to the car‑wash entrance** (≈ 15 seconds).  
4. **Enter the wash** and start the cycle (most modern washes let you pay with a card or app, so no need to stay in the car).  
5. **While the car is being cleaned**, you can:
   - Walk back to your house or office (if it’s a quick errand).  
   - Grab a coffee or snack nearby.  
   - Use the waiting time to stretch, check messages, etc.  
6. **When the wash finishes**, walk back to the car and drive away.

---

## If you decide to drive (e.g., bad weather)

1. **Start the engine** (a few seconds).  
2. **Drive slowly** (≈ 5 km/h) to keep control in the short distance.  
3. **Pull up as close as possible** to the wash entrance—ideally with the driver’s seat right next to the gate.  
4. **Turn off the engine** (or leave it on low‑idle if you need climate control).  
5. **Exit the car**, pay, and let the wash run.  
6. **Re‑enter** when the cycle ends and drive away.

---

### TL;DR

- **Walk** if the weather is okay, you have no mobility issues, and there’s a safe path.  
- **Drive** only if the weather or personal circumstances make walking uncomfortable or unsafe.  

Either way, the car wash is only a few steps away—so you’ll be in and out in under a minute regardless of your choice. Happy clean‑car! 🚗✨


effort = 'high'
   3.99s | reasoning tokens: 932 | output: 1729


**Short answer:** If the car wash is a drive‑through, you’ll have to drive the car in anyway, so you might as well just drive straight there. If you’re not already in the car (or you’d rather get a quick step‑in), walking the 50 m first is easy, healthy, and almost carbon‑free.

---

## Decision guide

| Factor | Walk (≈1 min) | Drive (≈30 sec) |
|--------|---------------|-----------------|
| **Convenience** | You have to get out, lock the car, walk to the wash, then hop back in. | One continuous motion; no extra steps. |
| **Weather** | Pleasant? Great for a quick stroll. Bad (rain, snow, extreme heat)? Better stay inside the car. |
| **Safety** | Safe if there’s a sidewalk or clear path. If it’s a busy road, watch traffic. | Safe if you can navigate the short turn without a lot of traffic. |
| **Energy & emissions** | Zero fuel use, a tiny bit of personal exercise. | Uses a few millilitres of fuel (≈0.1 km round‑trip) – essentially negligible, but still a small carbon footprint. |
| **Time** | ~1 minute walking + a few seconds to get back in the car. | ~30 seconds of driving; overall time difference is negligible. |
| **Kid/pet/gear** | Easier if you have strollers, groceries, or a pet that doesn’t like car rides. | Simpler if you’re already loaded with stuff and don’t want to lug it out. |
| **Car‑wash type** | If it’s a **self‑service** bay where you need to be inside the car, you’ll still drive in after walking. | If it’s an **automatic “drive‑through”** that pulls you in automatically, you must drive. |

---

## Practical tip‑by‑tip

1. **Check the wash’s entry style**  
   - *Drive‑through*: Just drive straight to the entrance.  
   - *Self‑service*: You can walk up, lock the car, then drive it in once you’re ready.

2. **Consider the weather**  
   - Warm, sunny day → enjoy the mini‑walk.  
   - Rainy, snowy, or very hot → stay in the car (or bring an umbrella).

3. **Think about what you have with you**  
   - If you’re carrying groceries, a stroller, or a pet, a quick walk may be less hassle than juggling them in a car.  
   - If you’re already seated with a coffee, the fastest route is to drive.

4. **Safety first**  
   - Make sure there’s a safe, well‑lit path.  
   - Lock the car while you’re away (or use a key‑less entry that auto‑locks).

5. **Environmental mindset**  
   - Walking 50 m uses zero fuel and adds a tiny bit of exercise—good for both you and the planet.  
   - Driving the extra 100 m round‑trip consumes only a few millilitres of gasoline or electricity, but if you’re aiming for the lowest possible footprint, the walk wins.

---

## Bottom line

- **If you need to be inside the car to get it washed** (most automatic washes), just drive straight there.  
- **If you’re currently not in the car** and the weather’s nice, a quick walk to the wash, lock the car, and then drive it in is a pleasant, low‑impact option.  

Either way, the time difference is negligible—choose the option that feels safest and most convenient for you right now. Safe cleaning! 🚗✨

**The trap:** the car is the thing being washed, so you have to **drive**. Walking 50
metres leaves the car at home. "Only 50 meters" is a distractor planted to buy a fast
"just walk, it's close."

With no reasoning step the model often anchors on the short distance. With `effort='high'`
the explicit reasoning step catches the dependency more reliably — and the token counts
above show what that reliability cost you.

**In production you tune effort per task:** low for lookups, high only where a wrong
answer is expensive. Paying for `high` on every call is how an LLM feature becomes
unaffordable.


<a id="pattern-4"></a>
## Pattern 4 — Structured output: a schema, not a hope

You can *ask* for JSON in the prompt. It works most of the time. "Most of the time" is
not good enough when the next line of your program is `json.loads()`.

**Structured outputs** make the schema a contract enforced at the API layer. You define
it once as a Pydantic model; you get back a typed Python object.

In [18]:
import json
from pydantic import BaseModel
from typing import List, Literal

class IncidentTriage(BaseModel):
    severity:    Literal["P0", "P1", "P2", "P3"]
    summary:     str
    first_step:  str
    secrets_involved: bool
    affected:    List[str]

print("This is the JSON schema the SDK sends to the API:")
print(json.dumps(IncidentTriage.model_json_schema(), indent=2)[:700] + "\n...")


This is the JSON schema the SDK sends to the API:
{
  "properties": {
    "severity": {
      "enum": [
        "P0",
        "P1",
        "P2",
        "P3"
      ],
      "title": "Severity",
      "type": "string"
    },
    "summary": {
      "title": "Summary",
      "type": "string"
    },
    "first_step": {
      "title": "First Step",
      "type": "string"
    },
    "secrets_involved": {
      "title": "Secrets Involved",
      "type": "boolean"
    },
    "affected": {
      "items": {
        "type": "string"
      },
      "title": "Affected",
      "type": "array"
    }
  },
  "required": [
    "severity",
    "summary",
    "first_step",
    "secrets_involved",
    "affected"
  ],
  "title": "IncidentTriage",
  "type": "obj
...


In [19]:
REPORT = """
Someone pushed a commit to our public GitHub repo at 14:05 that included a .env file
with a live OpenAI key and our Postgres connection string. It has been public for about
40 minutes. The repo has 12 forks. Our billing dashboard already shows an unusual spike.
"""

parsed = client.responses.parse(
    model        = MODEL,
    instructions = "You are an on-call incident triage assistant. Extract a structured triage record.",
    input        = REPORT,
    text_format  = IncidentTriage,
)

triage = parsed.output_parsed
print(f"type: {type(triage).__name__}\n")
print(f"severity         : {triage.severity}")
print(f"secrets involved : {triage.secrets_involved}")
print(f"summary          : {triage.summary}")
print(f"first step       : {triage.first_step}")
print(f"affected         : {triage.affected}")
print()
print("✅ A real typed object. No json.loads, no KeyError, no regex.")


type: IncidentTriage

severity         : P0
secrets involved : True
summary          : Live OpenAI API key and Postgres connection string were exposed in a public GitHub repo for ~40 minutes, causing a billing spike.
first step       : Immediately revoke the exposed OpenAI key and Postgres credentials, remove the .env file from the repo (force‑push), request GitHub to purge cached history, rotate all affected secrets, and monitor for unauthorized usage.
affected         : ['OpenAI API', 'PostgreSQL database', 'GitHub repository']

✅ A real typed object. No json.loads, no KeyError, no regex.


`triage` is a Python object with autocomplete and type checking. The schema becomes the
**contract between the prompt layer and the application layer** — any engineer on the
team can read it and know exactly what to expect. This is what replaces fragile parsing.


<a id="pattern-5"></a>
## Pattern 5 — Grounded context: "RAG by hand"

By default the model answers from training data. It knows nothing about your company,
your documents, or **this course**. The simplest fix is to paste the relevant document
into the instructions and tell it to use only that.

This is retrieval-augmented generation without the retrieval — and it is the direct
ancestor of everything in Session 2.

Below is a small set of facts about *this* repo. The model cannot know them, which makes
the weak→strong contrast real rather than staged.

In [20]:
# A tiny corpus, kept inline so this notebook has no external dependencies.
COURSE_FACTS = """
AIE11 COURSE SETUP FACTS

Python: projects require >=3.12,<3.15. The course currently pins 3.13.
Package and environment manager: uv. Each project folder has its own pyproject.toml,
  and you run `uv sync` inside that folder.
Node.js: version 24, used for frontend tooling and Vercel deployment.
Officially supported tools: VS Code and Claude Code. Cursor, Codex and others work,
  but you debug those yourself.
API keys: stored in a .env file in each PROJECT folder, never in your home directory
  and never committed. .env is gitignored.
Git model: three places. upstream (AI Makerspace, pull only), origin (your fork, you
  push here), local (your laptop, you edit here).
Branching: the course works directly on main. develop and feature branches are taught
  as the professional pattern but not used in coursework.
Structure: 10 weeks, 20 sessions. Session folders are numbered (01_, 02_, ...).
  Technical challenges live in 00_Technical_Challenges.
Agent behaviour: AGENTS.md at the repo root puts coding agents into teaching-assistant
  mode. They explain and debug, but decline to write assignment solutions.
"""

COURSE_Q = "In the AIE11 course, where exactly do I put my OpenAI key, and which Python version should I be on?"

print("=" * 60)
print("WEAK — no context. It will guess, plausibly and wrongly.")
print("=" * 60)
show(ask(COURSE_Q))


WEAK — no context. It will guess, plausibly and wrongly.


**Quick answer**

| What you need | Where to put it | Recommended Python version |
|---------------|----------------|----------------------------|
| **OpenAI API key** | In an environment variable (or a `.env` file) that your notebook / script reads at runtime | **Python 3.10** (any 3.9 + works, but the course materials are tested on 3.10) |

---

## 1️⃣ Where to put your OpenAI key

### A. Using an environment variable (the most portable way)

```bash
# Bash / Zsh (Linux / macOS)
export OPENAI_API_KEY="sk-XXXXXXXXXXXXXXXXXXXXXXXX"

# Windows PowerShell
$env:OPENAI_API_KEY = "sk-XXXXXXXXXXXXXXXXXXXXXXXX"
```

- **Why?** Keeps the secret out of your code and works both locally and on cloud notebooks (Colab, JupyterHub, etc.).
- **How to read it in Python**

```python
import os
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise RuntimeError("OpenAI key not found. Set OPENAI_API_KEY environment variable.")
```

### B. Using a `.env` file (convenient for local development)

1. Create a file called `.env` in the root of the project (same folder where you run `jupyter notebook` or `python script.py`).

   ```text
   # .env
   OPENAI_API_KEY=sk-XXXXXXXXXXXXXXXXXXXXXXXX
   ```

2. Install the helper package (once per environment):

   ```bash
   pip install python-dotenv
   ```

3. Load the variables at the start of your notebook / script:

   ```python
   from dotenv import load_dotenv
   import os

   load_dotenv()                # reads .env into os.environ
   openai_api_key = os.getenv("OPENAI_API_KEY")
   ```

### C. (Optional) A tiny `config.py` wrapper

If you prefer a single place to import settings, you can make a small module:

```python
# config.py
import os
from dotenv import load_dotenv

load_dotenv()                       # safe to call multiple times
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in the environment or .env file.")
```

Then just do:

```python
from config import OPENAI_API_KEY
```

> **Never** commit your key to Git! Add `.env` (or any file that contains the key) to `.gitignore`.

---

## 2️⃣ Which Python version should you use?

The AIE11 course materials were authored and tested with **Python 3.10**.  

- **Why 3.10?**  
  - All OpenAI SDK releases (the `openai` pip package) are fully compatible.  
  - The `typing` and `match‑case` features introduced in 3.10 are used in a few notebooks.  
  - The course’s auxiliary libraries (e.g., `pydantic`, `fastapi`, `jupyterlab`) have stable wheels for 3.10.

- **You can also use** any **3.9 +** version (3.11 works fine) as long as you:

  1. Verify that the notebook’s `requirements.txt` installs cleanly (`pip install -r requirements.txt`).  
  2. If you run into a syntax error related to the new `match` statement, you’re probably on < 3.10.

### Installing the right version

#### Using **conda** (recommended for beginners)

```bash
conda create -n aie11 python=3.10 -y
conda activate aie11
pip install -r requirements.txt   # or `conda install` for the listed packages
```

#### Using **pyenv** (if you prefer the system Python manager)

```bash
pyenv install 3.10.12          # latest 3.10 patch at the time of writing
pyenv virtualenv 3.10.12 aie11
pyenv activate aie11
pip install -r requirements.txt
```

#### Using plain **venv** (works everywhere)

```bash
python3.10 -m venv aie11-env
source aie11-env/bin/activate   # Windows: aie11-env\Scripts\activate
pip install -r requirements.txt
```

---

## 3️⃣ TL;DR Checklist (copy‑paste into a terminal)

```bash
# 1️⃣ Choose a place to store the key
echo "OPENAI_API_KEY=sk-XXXXXXXXXXXXXXXXXXXXXXXX" > .env
# (or set the env var manually)

# 2️⃣ Create a clean Python 3.10 environment
python3.10 -m venv aie11-env
source aie11-env/bin/activate   # Windows: aie11-env\Scripts\activate

# 3️⃣ Install course dependencies
pip install -U pip setuptools wheel
pip install -r requirements.txt
pip install python-dotenv   # only needed if you use the .env approach

# 4️⃣ Verify everything works
python - <<'PY'
import os, openai
from dotenv import load_dotenv
load_dotenv()
key = os.getenv("OPENAI_API_KEY")
print("Key loaded?" , bool(key))
# quick API test (will raise if key missing/invalid)
# openai.api_key = key
# print(openai.Models.list())
PY
```

If the script prints `Key loaded? True` (and the optional API test succeeds), you’re ready to dive into the AIE11 notebooks! 🚀

---

### Need more help?

- **JupyterLab/Colab** – just add a cell at the top:

  ```python
  %load_ext dotenv
  %dotenv .env
  ```

- **VS Code** – enable the “Python: Activate Environment” command after you create the venv.

Happy coding, and enjoy the AIE11 course! If you hit any specific error (e.g., “module not found” or “SyntaxError: invalid syntax”), drop the exact traceback and I’ll walk you through it.

In [21]:
GROUNDED = (
    "You answer questions about the AIE11 course setup using ONLY the facts below. "
    "If the answer is not in the facts, say so explicitly rather than guessing.\n\n"
    + COURSE_FACTS
)

print("=" * 60)
print("STRONG — facts pasted into instructions")
print("=" * 60)
show(ask(COURSE_Q, instructions=GROUNDED))


STRONG — facts pasted into instructions


**Where to put your OpenAI key**

- Create (or edit) a **`.env` file inside the specific project folder** you are working on.  
- Add your OpenAI key there, e.g.:

```text
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

- The `.env` file is **git‑ignored**, so it will never be committed, and it should **not** be placed in your home directory.

**Which Python version to use**

- All AIE11 projects require a Python version **≥ 3.12 and < 3.15**.  
- The course currently **pins Python 3.13**, so that is the version you should be using (it satisfies the required range).

So, put the key in a `.env` file in your project directory, and run the project with Python 3.13.

The grounded answer names the project folder and the version range, because it read them.
The ungrounded one produced confident, generic advice.

Now the half of grounding people forget — **can it say "I don't know"?**


In [22]:
OUT_OF_SCOPE = "What is the refund policy for the AIE11 bootcamp?"

print("Nothing in COURSE_FACTS mentions refunds. A refusal is the CORRECT answer here.")
print("=" * 60)
show(ask(OUT_OF_SCOPE, instructions=GROUNDED))


Nothing in COURSE_FACTS mentions refunds. A refusal is the CORRECT answer here.


I’m sorry, but the refund policy for the AIE11 bootcamp isn’t covered in the facts I have.

> ⚠️ **If that just answered anyway, you have found the single most common bug in
> deployed RAG systems.** The retrieved context did not contain the answer, and the model
> filled the gap from training data without telling you. Nothing in the pipeline noticed.
>
> This pattern scales up to any document that fits in the context window. For thousands
> of documents — or documents that change — you need real retrieval: chunking, embedding,
> vector search. **That is Session 2**, and this is the problem it exists to solve.


---

<a id="part-4"></a>
# Part 4 — Frontend and backend

<a id="why-split"></a>
## Why split them at all?

Everything above runs in a notebook. A notebook is a fine lab and a terrible product:
it has no users, no URL, and it holds your API key in the same process as your UI.

That last point is the whole reason for the split.

Picture shipping the notebook logic straight into a web page. The `OPENAI_API_KEY` would
have to go with it — into JavaScript the browser downloads. **Anything in the browser
belongs to the user**, including your key. One "view source" and your billing is theirs.

So the work divides by *who is allowed to know what*:

| | **Frontend** | **Backend** |
|---|---|---|
| Runs on | the user's device | your server |
| Written in | TypeScript / React | Python (FastAPI) |
| Knows your API key | **never** | yes |
| Job | collect input, render output | hold secrets, call the LLM, enforce rules |
| If a user inspects it | they see everything | they see nothing |
| Deployed to | Vercel | Vercel / Render |

The flow is one hop longer than a notebook, and the extra hop is the security boundary:

```
browser  ──request──>  your backend  ──API key──>  OpenAI
   ▲                        │
   └──────── answer ────────┘
```

Your system prompt also lives on the backend. If it shipped to the browser, every user
could read it — and edit it. **Rules only hold where the user cannot reach them.**

<a id="how-no-demo"></a>
## How — no demo, by design

We are not building this today; Session 1 is about having a machine that works. The
reference implementation is **[The AI Engineer Challenge](https://github.com/AI-Maker-Space/The-AI-Engineer-Challenge/tree/dc-challenge)** —
and the machine you validated in Part 0 runs it *as-is*: same `uv sync`, same Python 3.13,
same per-project `.env`. Nothing new to install.

It is this exact split, made literal:

| Layer | In the repo | Who holds the key |
|---|---|---|
| Frontend | `frontend/` — **empty on purpose; you vibe-code it** | never |
| Backend | `api/index.py` — **provided**, FastAPI, `POST /api/chat` + `GET /api/health` | yes |
| LLM | OpenAI **or** Anthropic, via one `LLM_PROVIDER` switch | — |
| Deployment | `npx vercel` from the repo root — both halves, one domain | server-side only |

The key and the system prompt live in the backend you were given; the frontend you build
calls `POST /api/chat` and never sees either. That is the whole lesson above, as running code.

The path through it:

1. **Git** — fork, clone, set `upstream`. Part 1 of this notebook
2. **LLM API** — the call, the roles, the system prompt. Parts 2 and 3
3. **IDE** — VS Code plus Claude Code, pointed at the repo folder
4. **Backend (provided)** — open `api/index.py`; the key comes from `.env`, the system prompt is right there
5. **Frontend (you build)** — vibe-code a UI that talks to *your* `/api/chat`, never to OpenAI
6. **Deploy** — `npx vercel` from the root, Vercel builds both, the URL is real

> 🧭 **Same `AGENTS.md`, opposite instructions.** The Challenge repo carries an `AGENTS.md`
> too — but it tells the agent to *build and commit*, not to hold back like this course's
> teaching-assistant one. Same file, different context, different rules: the Part 2 system
> role, one more time.

Where to go for each:

- [The AI Engineer Challenge repo](https://github.com/AI-Maker-Space/The-AI-Engineer-Challenge/tree/dc-challenge) — the code (backend provided, frontend yours)
- [aimakerspace.io/aie-challenge](https://aimakerspace.io/aie-challenge) — the challenge brief
- [Onramp Session 02](../00_Docs/Prerequisites/The_AI_Engineer_Onramp/Session_02_Back_End_Web_App_Development_&_Deployment_of_LLM_Applications/README.md) — backend and deployment
- [Onramp Session 03](../00_Docs/Prerequisites/The_AI_Engineer_Onramp/Session_03_Connecting_Frontend_UIs_to_Backend_Deployments/README.md) — wiring a frontend to a deployed backend
- [Onramp Session 04](../00_Docs/Prerequisites/The_AI_Engineer_Onramp/Session_04_Bulding_Complex_End_to_End_LLM_Applications/README.md) — the complex end-to-end version

> ℹ️ The repo links point at the **`dc-challenge`** branch, where the updated
> provider-agnostic version lives. They move to `main` once it is merged.

---

<a id="activity"></a>
## 🏗️ Activity — stack three patterns

Each pattern above works alone. The leverage is in combining them.

**Your task:** one call that answers a question about this course using at least three
patterns together — a **persona**, the **grounded context** from `COURSE_FACTS`, and a
**structured output** schema. Then compare it against the naive one-liner from Pattern 5.

The scaffold is below. Fill in the two TODOs.

In [23]:
from pydantic import BaseModel
from typing import List

class SetupAnswer(BaseModel):
    answer:        str          # 2-3 sentences, grounded in COURSE_FACTS
    commands:      List[str]    # exact commands the student should run, in order
    answered_from_context: bool # False if COURSE_FACTS did not cover it
    gotcha:        str          # the mistake people make here

STACKED_INSTRUCTIONS = (
    # TODO 1: your persona. Who should this sound like?
    "You are ... "
    # TODO 2: ground it. Tell the model to use ONLY the facts, and to set
    #         answered_from_context=False when they don't cover the question.
    "\n\nFACTS:\n" + COURSE_FACTS
)

result = client.responses.parse(
    model        = MODEL,
    instructions = STACKED_INSTRUCTIONS,
    input        = "I just cloned the repo. How do I get 02_Dense_Vector_Retrieval running?",
    text_format  = SetupAnswer,
)

out = result.output_parsed
print(f"from context : {out.answered_from_context}")
print(f"answer       : {out.answer}\n")
print("commands:")
for i, c in enumerate(out.commands, 1):
    print(f"  {i}. {c}")
print(f"\ngotcha       : {out.gotcha}")


from context : True
answer       : 1. **Install the correct Python version** – The project is pinned to Python 3.13 (>=3.12,<3.15). Use pyenv, asdf, or your system installer to get Python 3.13 and make it the active interpreter for the repo.
2. **Install uv** – If you don’t have the package manager yet, run `pip install uv` (or follow the official uv install guide for your OS).
3. **Create the .env file** – Inside the `02_Dense_Vector_Retrieval` folder create a `.env` with the keys the code expects (e.g., `OPENAI_API_KEY=your-key`). The file is git‑ignored, so it won’t be committed.
4. **Install the project dependencies** – From the root of the `02_Dense_Vector_Retrieval` folder run:
   ```
   uv sync
   ```
   This reads the `pyproject.toml` in that folder and creates an isolated virtual environment.
5. **Run the demo / script** – The session usually provides an entry‑point like `main.py` or a module. Use uv to execute it:
   ```
   uv run python -m dense_vector_retrieval   # or uv ru

**Then test the refusal path.** Re-run the cell with an input `COURSE_FACTS` cannot
answer, such as `"What time does Thursday's session start?"`. Does `answered_from_context`
come back `False`? If it comes back `True` with an invented time, you have reproduced the
Session 2 problem in your own code — which is the most useful thing you could learn today.


---

<a id="done-when"></a>
## ✅ Done when

- [ ] Every cell in Part 0 prints ✅ (or you know exactly which one doesn't and why)
- [ ] `git remote -v` shows both `origin` and `upstream`
- [ ] You made one real LLM API call and read its token counts
- [ ] You ran all five patterns and saw the weak→strong difference in each
- [ ] The refusal cell in Pattern 5 either refused, or you understand why it didn't
- [ ] The stacked activity runs with your own persona and grounding
- [ ] The installer's own check reports **All good** —
      `bash ~/ai-eng/setup.sh --check` (Mac/Linux) or
      `powershell -ExecutionPolicy Bypass -File $HOME\ai-eng\setup.ps1 -Check` (Windows)